In [ ]:
import json
import os
import networkx as nx
from collections import defaultdict
from datetime import datetime

def get_fingerprint(alert):
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

In [ ]:
def get_session_groups(alerts, gap_sec=120):
    if not alerts:
        return []
   
    sorted_alerts = sorted(alerts, key=lambda x: x['ts'])
    sessions = [[sorted_alerts[0]]]
    
    for current_alert in sorted_alerts[1:]:
        current_ts = datetime.fromisoformat(current_alert['ts'].replace('Z', '+00:00'))
        last_ts = datetime.fromisoformat(sessions[-1][-1]['ts'].replace('Z', '+00:00'))

        if (current_ts - last_ts).total_seconds() <= gap_sec:
            sessions[-1].append(current_alert)
        else:
            sessions.append([current_alert])
            
    return sessions

In [ ]:
def build_directed_graph(services_data):
    G = nx.DiGraph()
    
    for svc in services_data['services']:
        G.add_node(svc['name'])
    for store in services_data['stores']:
        G.add_node(store['name'])
    for edge in services_data['edges']:
        G.add_edge(edge['from'], edge['to'])
        
    return G

def topology_grouping_directed(session_alerts, directed_graph, max_hop=2):
    if not session_alerts:
        return []

    clean_session_alerts = []
    orphan_groups = []
    
    for alert in session_alerts:
        note = alert.get('labels', {}).get('note', '').lower()
        if 'unrelated' in note or 'noise' in note or 'independent' in note:
            orphan_groups.append([alert])
        else:
            clean_session_alerts.append(alert)
            
    if not clean_session_alerts:
        return orphan_groups

    service_to_alerts = defaultdict(list)
    for alert in clean_session_alerts:
        service_to_alerts[alert['service']].append(alert)
        
    unique_services = list(service_to_alerts.keys())
    parent = {s: s for s in unique_services}
    
    def find(x):
        if parent[x] == x: return x
        parent[x] = find(parent[x])
        return parent[x]
        
    def union(x, y):
        root_x, root_y = find(x), find(y)
        if root_x != root_y: parent[root_x] = root_y

    for i in range(len(unique_services)):
        for j in range(i + 1, len(unique_services)):
            s1, s2 = unique_services[i], unique_services[j]

            connected = False
            try:
                if nx.has_path(directed_graph, s1, s2) and nx.shortest_path_length(directed_graph, s1, s2) <= max_hop:
                    connected = True
                elif nx.has_path(directed_graph, s2, s1) and nx.shortest_path_length(directed_graph, s2, s1) <= max_hop:
                    connected = True
            except nx.NetworkXNoPath:
                pass
                
            if connected:
                union(s1, s2)

    topo_clusters = defaultdict(list)
    for s in unique_services:
        topo_clusters[find(s)].extend(service_to_alerts[s])

    final_groups = list(topo_clusters.values()) + orphan_groups
    return final_groups

In [13]:
def run_pipeline():
    with open('dataset/services.jsonl', 'r', encoding='utf-8') as f:
        services_data = json.load(f)
        
    alerts = []
    with open('dataset/alerts_sample.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                alerts.append(json.loads(line.strip()))

    directed_graph = build_directed_graph(services_data)

    sessions = get_session_groups(alerts, gap_sec=120)
    
    final_clusters = []
    cluster_counter = 0

    for s_idx, session_alerts in enumerate(sessions):
        topo_groups = topology_grouping_directed(session_alerts, directed_graph, max_hop=2)
        
        for group in topo_groups:
            cluster_id = f"c-{s_idx:03d}-{cluster_counter:03d}"
            cluster_counter += 1
            
            services = sorted(list(set(a['service'] for a in group)))
            fingerprints = sorted(list(set(get_fingerprint(a) for a in group)))
            timestamps = [a['ts'] for a in group]
            severities = [a['severity'] for a in group]

            sev_priority = {"warn": 1, "crit": 2}
            max_sev = "warn" if max(sev_priority[s] for s in severities) == 1 else "crit"
            
            final_clusters.append({
                "cluster_id": cluster_id,
                "alert_count": len(group),
                "services": services,
                "time_range": [min(timestamps), max(timestamps)],
                "max_severity": max_sev,
                "fingerprints": fingerprints
            })

    input_alerts_count = len(alerts)
    output_clusters_count = len(final_clusters)
    reduction_ratio = 1.0 - (output_clusters_count / input_alerts_count)
    
    output_json = {
        "input_alerts": input_alerts_count,
        "output_clusters": output_clusters_count,
        "reduction_ratio": round(reduction_ratio, 2),
        "clusters": final_clusters
    }

    os.makedirs('results', exist_ok=True)
    with open('results/cluster_summary.json', 'w', encoding='utf-8') as f:
        json.dump(output_json, f, indent=2, ensure_ascii=False)
        
    print(f"-> Tổng số lượng alert đầu vào: {input_alerts_count}")
    print(f"-> Số lượng cụm sự cố: {output_clusters_count}")
    print(f"-> Tỉ lệ tinh giản nhiễu hệ thống: {reduction_ratio * 100:.2f}%")

run_pipeline()

-> Tổng số lượng alert đầu vào: 20
-> Số lượng cụm sự cố: 3
-> Tỉ lệ tinh giản nhiễu hệ thống: 85.00%
